# Fine-Tuning

This notebook fine-tunes a supervised target on the bundled Breast Cancer Wisconsin dataset. The example reshapes selected columns into a nested `measurements` array so the model sees repeated measurement objects rather than one wide flat row.

The imports mirror the training tutorial. The dataset is loaded locally from scikit-learn and then converted into JSON-like records.

In [1]:
import contextlib
import io
import logging

import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint
from sklearn.datasets import load_breast_cancer

import json2vec as j2v

logger.remove()

Each record contains a list of measurement dictionaries plus a diagnosis label. This is intentionally small, but it demonstrates the pattern used for orders with line items, sessions with events, or entities with repeated attributes.

In [2]:
ROOT = "[*][*]"
cancer = load_breast_cancer()
measurement_names = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
feature_index = {name: list(cancer.feature_names).index(name) for name in measurement_names}
malignant = [index for index, target in enumerate(cancer.target) if target == 0][:16]
benign = [index for index, target in enumerate(cancer.target) if target == 1][:16]
indices = [*malignant, *benign]

records = pl.DataFrame(
    [
        {
            "measurements": [
                {"name": name.replace(" ", "_"), "value": float(cancer.data[index, column])}
                for name, column in feature_index.items()
            ],
            "diagnosis": cancer.target_names[int(cancer.target[index])],
        }
        for index in indices
    ],
    strict=False,
)

records.head()

measurements,diagnosis
list[struct[2]],str
"[{""mean_radius"",17.99}, {""mean_texture"",10.38}, … {""mean_smoothness"",0.1184}]","""malignant"""
"[{""mean_radius"",20.57}, {""mean_texture"",17.77}, … {""mean_smoothness"",0.08474}]","""malignant"""
"[{""mean_radius"",19.69}, {""mean_texture"",21.25}, … {""mean_smoothness"",0.1096}]","""malignant"""
"[{""mean_radius"",11.42}, {""mean_texture"",20.38}, … {""mean_smoothness"",0.1425}]","""malignant"""
"[{""mean_radius"",20.29}, {""mean_texture"",14.34}, … {""mean_smoothness"",0.1003}]","""malignant"""


The nested `Array` defines the measurement context. Inside that array, `name` identifies the measurement and `value` carries the numeric signal. The root-level `diagnosis` field is the fine-tuning target.

In [3]:
model = j2v.Model.from_schema(
    j2v.Array(
        j2v.Category("name", query=f"{ROOT}.measurements[*].name", max_vocab_size=16),
        j2v.Number("value", query=f"{ROOT}.measurements[*].value"),
        name="measurements",
        max_length=len(measurement_names),
    ),
    j2v.Category("diagnosis", query=f"{ROOT}.diagnosis", target=True, max_vocab_size=2),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)
_ = model.set(j2v.where("name") == "record", embed=True)

The data module does not need special nested-data code. The schema queries describe where values live, and the data module handles batching and tensorization.

In [4]:
datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

Fine-tuning here is intentionally minimal: the notebook proves the schema shape and training path, not benchmark performance.

In [5]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    logging.disable(logging.CRITICAL)
    try:
        lit.seed_everything(7, workers=True, verbose=False)
        trainer = lit.Trainer(
            accelerator="cpu",
            max_epochs=1,
            logger=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            enable_checkpointing=False,
            limit_train_batches=1,
            limit_val_batches=1,
        )
        trainer.fit(model=model, datamodule=datamodule)
    finally:
        logging.disable(logging.NOTSET)

The plot is useful for nested schemas because it shows which modules belong to the root record and which belong to the repeated measurement context.

In [6]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  24,935                                │
│  Schema map                                                            arrays  2                                     │
│                                                                        fields  3                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               

Prediction decodes only configured targets. In this case, the output is the model response for `record/diagnosis`.

In [7]:
batch = [[record] for record in records.to_dicts()[:3]]
pprint(model.predict(batch))

{
│   'record/diagnosis': {
│   │   'state': {
│   │   │   'valued': [0.1650972068309784, 0.1650972068309784, 0.1650972217321396],
│   │   │   'null': [0.06980018317699432, 0.06980018317699432, 0.06980018317699432],
│   │   │   'padded': [0.6656454205513, 0.6656454205513, 0.6656454205513],
│   │   │   'masked': [0.05628220736980438, 0.05628220736980438, 0.05628220736980438],
│   │   │   'other': [0.0431748628616333, 0.0431748628616333, 0.0431748628616333]
│   │   },
│   │   'content': {'value': [None, None, None], 'probability': [0.0, 0.0, 0.0], 'topk': [[], [], []]}
│   }
}